# 🧠 Smart Library — Enrichissement des métadonnées techniques
## Notebook 08 : Data Catalog augmenté par LLM (Groq)

---

### Objectif
S'appuyer sur le **schéma technique réel** de chaque table Iceberg (bronze,
silver, gold) — colonnes, types, nullabilité, partitionnement, échantillon de
données — pour générer automatiquement, via un LLM hébergé sur **Groq**
(inférence très rapide sur LPU), une **documentation métier enrichie** :

- une description de la table (à quoi elle sert, dans quelle couche Medallion)
- une description de chaque colonne
- un tag de gouvernance par colonne (`PII` / `métier` / `technique`) utile pour
  la thèse (partie gouvernance des données)

Ces métadonnées sont ensuite **écrites dans le catalogue Iceberg lui-même**
(`COMMENT` de table et de colonne, via Spark SQL) — donc visibles par n'importe
quel moteur qui se connecte au catalogue REST Lakekeeper (Spark, Trino, DuckDB...),
sans dépendre de ce notebook. Un **data dictionary** (Markdown + JSON) est en
plus exporté pour la documentation / l'annexe de la thèse.

### Prérequis
1. Un compte gratuit sur [console.groq.com](https://console.groq.com/keys) et une clé API.
2. La variable d'environnement `GROQ_API_KEY` définie **avant** de lancer Jupyter
   (ajoutée dans `docker-compose.yml`, service `jupyter` → `environment:`), ou
   à défaut décommenter la ligne `os.environ["GROQ_API_KEY"] = "..."` ci-dessous
   (à ne jamais committer avec une vraie clé).
3. Avoir exécuté au moins `01_ingestion_bronze.ipynb` (et idéalement 02/03) pour
   que des tables existent réellement dans le catalogue.

### Pourquoi Groq et pas un appel Claude/OpenAI classique ?
Ce notebook fait **des dizaines de petits appels** (un par table, potentiellement
un par colonne) : la latence très faible de Groq (LPU, modèles ouverts type
Llama / GPT-OSS) rend l'enrichissement de tout un catalogue quasi instantané,
ce qui serait plus lent/coûteux avec un appel API classique par colonne.


---
## Étape 0 — Initialisation SparkSession + client Groq

In [1]:
!pip install --quiet groq

import os, sys, time, re, json
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

spark = (
    SparkSession.builder
    .appName("SmartLibrary - Enrichissement Metadonnees")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")

# ── Client Groq ────────────────────────────────────────────────
from groq import Groq

# os.environ["GROQ_API_KEY"] = "gsk_xxx"  # ⚠️ à décommenter seulement en local, jamais à committer

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "⚠️  GROQ_API_KEY absente. Crée une clé gratuite sur "
        "https://console.groq.com/keys et exporte-la avant de lancer Jupyter, "
        "ou décommente la ligne os.environ ci-dessus (dev uniquement)."
    )

groq_client = Groq(api_key=GROQ_API_KEY)

# Modèle rapide/gratuit sur Groq. llama-3.3-70b-versatile est en cours de
# dépréciation côté Groq (2026) : on utilise son remplaçant recommandé.
# Vérifier la liste à jour sur https://console.groq.com/docs/models si besoin.
MODEL_GROQ = "openai/gpt-oss-120b"

print(f"✅ Client Groq initialisé — modèle : {MODEL_GROQ}")


✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
✅ Client Groq initialisé — modèle : openai/gpt-oss-120b


---
## Étape 1 — Inventaire technique : lister les tables et extraire leur schéma

On s'appuie uniquement sur ce que le catalogue Iceberg sait déjà (pas besoin
de deviner) : `SHOW TABLES`, `spark.table(...).schema`, les propriétés de
partitionnement et un petit échantillon de lignes pour donner du contexte au LLM.


In [2]:
def lister_tables(spark, namespace):
    """Retourne la liste des tables (nom complet catalogue.namespace.table) d'un namespace."""
    lignes = spark.sql(f"SHOW TABLES IN {namespace}").collect()
    return [f"{namespace}.{r['tableName']}" for r in lignes]


def extraire_schema_technique(spark, table_name):
    """
    Construit une fiche technique de la table à partir du catalogue Iceberg :
    colonnes (nom, type, nullable), colonnes de partition, et un échantillon
    de valeurs réelles (utile au LLM pour deviner le sens métier d'une colonne
    comme 'FR', 'EN'... ou un code de statut).
    """
    df = spark.table(table_name)

    colonnes = [
        {"nom": f.name, "type": f.dataType.simpleString(), "nullable": f.nullable}
        for f in df.schema.fields
    ]

    try:
        partitions = [
            row["name"] for row in
            spark.sql(f"DESCRIBE TABLE {table_name}").collect()
            if row["col_name"] == "# Partition Information"
        ]
    except Exception:
        partitions = []

    try:
        partition_cols = [
            r["name"] for r in spark.sql(f"SHOW PARTITIONS {table_name}").limit(0).collect()
        ]
    except Exception:
        partition_cols = []

    try:
        echantillon = df.limit(3).toPandas().to_dict(orient="records")
    except Exception:
        echantillon = []

    try:
        nb_lignes = df.count()
    except Exception:
        nb_lignes = None

    return {
        "table": table_name,
        "colonnes": colonnes,
        "nb_lignes": nb_lignes,
        "echantillon": echantillon,
    }


# Test rapide sur une table bronze
exemple = extraire_schema_technique(spark, f"{NS_BRONZE}.livres")
print(json.dumps(exemple, indent=2, ensure_ascii=False, default=str)[:1200])


{
  "table": "lakehouse.bronze.livres",
  "colonnes": [
    {
      "nom": "auteur",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "livre_id",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "date_publication",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "description",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "disponible",
      "type": "boolean",
      "nullable": true
    },
    {
      "nom": "editeur",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "genre",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "isbn",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "langue",
      "type": "string",
      "nullable": true
    },
    {
      "nom": "nb_avis",
      "type": "bigint",
      "nullable": true
    },
    {
      "nom": "nb_pages",
      "type": "bigint",
      "nullable": true
    },
    {
  

---
## Étape 2 — Génération des descriptions via Groq

Un seul appel LLM par table : on lui donne le schéma + un échantillon, et on
lui impose une réponse **JSON strict** (table + colonnes) pour pouvoir
l'appliquer automatiquement sans parsing fragile.


In [3]:
def construire_prompt(fiche_technique, couche):
    """Construit le prompt envoyé à Groq à partir de la fiche technique d'une table."""
    return f"""Tu es un data steward qui documente un data lakehouse (architecture
Medallion Bronze/Silver/Gold, Apache Iceberg) pour une bibliothèque en ligne.

Voici le schéma technique réel de la table "{fiche_technique['table']}"
(couche {couche}) :

Colonnes : {json.dumps(fiche_technique['colonnes'], ensure_ascii=False)}
Nombre de lignes : {fiche_technique['nb_lignes']}
Échantillon de données : {json.dumps(fiche_technique['echantillon'], ensure_ascii=False, default=str)[:1500]}

Réponds STRICTEMENT en JSON (aucun texte avant/après), avec ce format exact :
{{
  "description_table": "1 à 2 phrases en français décrivant le rôle métier de la table",
  "colonnes": {{
     "nom_colonne_1": {{"description": "...", "sensibilite": "PII|metier|technique"}},
     "nom_colonne_2": {{"description": "...", "sensibilite": "PII|metier|technique"}}
  }}
}}

Règles :
- "sensibilite" = "PII" si la colonne contient une donnée personnelle identifiante
  (nom, email, adresse...), "technique" pour les colonnes internes (préfixées `_`,
  `updated_at`, colonnes de partition), sinon "metier".
- Une entrée par colonne listée ci-dessus, aucune colonne inventée.
- Descriptions courtes (une phrase), en français.
"""


def generer_metadonnees_llm(fiche_technique, couche, essais=3):
    """Appelle Groq pour générer la documentation d'une table, avec retry simple."""
    prompt = construire_prompt(fiche_technique, couche)

    derniere_erreur = None
    for tentative in range(essais):
        try:
            reponse = groq_client.chat.completions.create(
                model=MODEL_GROQ,
                messages=[
                    {"role": "system", "content": "Tu réponds uniquement en JSON valide, sans commentaire."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
                response_format={"type": "json_object"},
            )
            contenu = reponse.choices[0].message.content
            return json.loads(contenu)
        except Exception as e:
            derniere_erreur = e
            time.sleep(1.5 * (tentative + 1))

    print(f"⚠️  Échec génération LLM pour {fiche_technique['table']} : {derniere_erreur}")
    return None


# Test sur la table d'exemple
meta_exemple = generer_metadonnees_llm(exemple, "bronze")
print(json.dumps(meta_exemple, indent=2, ensure_ascii=False))


{
  "description_table": "Table bronze contenant les enregistrements bruts des livres ingérés depuis les sources, avant toute transformation.",
  "colonnes": {
    "auteur": {
      "description": "Nom de l'auteur du livre",
      "sensibilite": "PII"
    },
    "livre_id": {
      "description": "Identifiant unique du livre dans le système",
      "sensibilite": "metier"
    },
    "date_publication": {
      "description": "Date de publication du livre",
      "sensibilite": "metier"
    },
    "description": {
      "description": "Résumé ou description du contenu du livre",
      "sensibilite": "metier"
    },
    "disponible": {
      "description": "Indique si le livre est actuellement disponible à la vente",
      "sensibilite": "metier"
    },
    "editeur": {
      "description": "Nom de la maison d'édition du livre",
      "sensibilite": "metier"
    },
    "genre": {
      "description": "Genre littéraire du livre",
      "sensibilite": "metier"
    },
    "isbn": {
      "d

---
## Étape 3 — Application des métadonnées dans le catalogue Iceberg

`ALTER TABLE ... SET TBLPROPERTIES('comment' = ...)` documente la table,
`ALTER TABLE ... ALTER COLUMN ... COMMENT ...` documente chaque colonne.
Ces métadonnées sont stockées côté catalogue Lakekeeper — elles resteront
visibles pour n'importe quel moteur (Spark, Trino, DuckDB) qui se connecte
au même catalogue REST.


In [4]:
def echapper_sql(texte):
    """Échappe les apostrophes pour une insertion sûre dans du SQL littéral."""
    return (texte or "").replace("'", "\\'")


def appliquer_metadonnees(spark, table_name, metadonnees):
    """Écrit la description de table + les commentaires de colonnes dans le catalogue Iceberg."""
    if not metadonnees:
        return 0

    description_table = echapper_sql(metadonnees.get("description_table", ""))
    if description_table:
        spark.sql(f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES ('comment' = '{description_table}')
        """)

    colonnes_maj = 0
    colonnes_reelles = {f.name for f in spark.table(table_name).schema.fields}

    for nom_colonne, infos in metadonnees.get("colonnes", {}).items():
        if nom_colonne not in colonnes_reelles:
            continue  # on ignore toute colonne halluciné par le LLM
        description = infos.get("description", "")
        sensibilite = infos.get("sensibilite", "metier")
        commentaire = echapper_sql(f"[{sensibilite}] {description}")
        try:
            spark.sql(f"""
                ALTER TABLE {table_name}
                ALTER COLUMN {nom_colonne}
                COMMENT '{commentaire}'
            """)
            colonnes_maj += 1
        except Exception as e:
            print(f"   ⚠️  Colonne {table_name}.{nom_colonne} non commentée : {e}")

    return colonnes_maj


# Application sur la table d'exemple
n = appliquer_metadonnees(spark, f"{NS_BRONZE}.livres", meta_exemple)
print(f"✅ {n} colonnes commentées sur {NS_BRONZE}.livres")


✅ 23 colonnes commentées sur lakehouse.bronze.livres


---
## Étape 4 — Boucle globale : enrichir tout le catalogue (bronze → silver → gold)

⏱️ Grâce à la vitesse d'inférence de Groq, enrichir toutes les tables du
lakehouse ne prend que quelques secondes malgré un appel LLM par table.


In [5]:
resultats = []  # data dictionary final, exporté à l'étape suivante

for namespace, nom_couche in [(NS_BRONZE, "bronze"), (NS_SILVER, "silver"), (NS_GOLD, "gold")]:
    tables = lister_tables(spark, namespace)
    print(f"\n=== Couche {nom_couche.upper()} — {len(tables)} table(s) ===")

    for table_name in tables:
        try:
            fiche = extraire_schema_technique(spark, table_name)
            meta  = generer_metadonnees_llm(fiche, nom_couche)
            n_col = appliquer_metadonnees(spark, table_name, meta)

            resultats.append({
                "table": table_name,
                "couche": nom_couche,
                "nb_lignes": fiche["nb_lignes"],
                "description": meta.get("description_table") if meta else None,
                "colonnes": meta.get("colonnes") if meta else {},
            })
            print(f"  ✅ {table_name:<45} {n_col} colonne(s) documentée(s)")
        except Exception as e:
            print(f"  ❌ {table_name:<45} erreur : {e}")

        time.sleep(0.3)  # confort rate-limit, largement sous les quotas Groq

print(f"\n📊 {len(resultats)} tables enrichies au total")



=== Couche BRONZE — 11 table(s) ===
  ✅ lakehouse.bronze.utilisateurs                 16 colonne(s) documentée(s)
  ✅ lakehouse.bronze.livres                       23 colonne(s) documentée(s)
  ✅ lakehouse.bronze.catalogue                    13 colonne(s) documentée(s)
  ✅ lakehouse.bronze.commandes                    19 colonne(s) documentée(s)
  ✅ lakehouse.bronze.stock_cow                    14 colonne(s) documentée(s)
  ✅ lakehouse.bronze.stock_mor                    14 colonne(s) documentée(s)
  ✅ lakehouse.bronze.factures                     12 colonne(s) documentée(s)
  ✅ lakehouse.bronze.evenements_clics             10 colonne(s) documentée(s)
  ✅ lakehouse.bronze.evenements_recherches        11 colonne(s) documentée(s)
  ✅ lakehouse.bronze.evenements_panier            10 colonne(s) documentée(s)
  ✅ lakehouse.bronze.evenements_avis              10 colonne(s) documentée(s)

=== Couche SILVER — 6 table(s) ===
  ✅ lakehouse.silver.utilisateurs                 18 colonne(s) docum

---
## Étape 5 — Export du data dictionary (documentation / annexe de thèse)

On exporte à la fois un JSON exploitable par un script et un Markdown lisible,
générés à partir des **mêmes métadonnées** que celles écrites dans le catalogue
Iceberg (donc toujours synchronisés avec la réalité technique des tables).


In [6]:
DOSSIER_DOCS = Path("/home/jovyan/data/catalogue_donnees")
DOSSIER_DOCS.mkdir(parents=True, exist_ok=True)

# ── Export JSON ────────────────────────────────────────────────
chemin_json = DOSSIER_DOCS / "data_dictionary.json"
with open(chemin_json, "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2, default=str)

# ── Export Markdown ───────────────────────────────────────────
lignes_md = ["# 📖 Data Dictionary — Smart Library Lakehouse\n",
             f"_Généré automatiquement via Groq ({MODEL_GROQ}) le {datetime.now():%Y-%m-%d %H:%M}_\n"]

for couche in ["bronze", "silver", "gold"]:
    tables_couche = [r for r in resultats if r["couche"] == couche]
    if not tables_couche:
        continue
    lignes_md.append(f"\n## Couche {couche.capitalize()}\n")
    for r in tables_couche:
        lignes_md.append(f"\n### `{r['table']}`\n")
        lignes_md.append(f"{r['description'] or '_(non généré)_'}\n")
        lignes_md.append(f"\n_{r['nb_lignes']} ligne(s)_\n")
        lignes_md.append("\n| Colonne | Sensibilité | Description |\n|---|---|---|\n")
        for nom_col, infos in (r["colonnes"] or {}).items():
            lignes_md.append(f"| `{nom_col}` | {infos.get('sensibilite','-')} | {infos.get('description','-')} |\n")

chemin_md = DOSSIER_DOCS / "data_dictionary.md"
with open(chemin_md, "w", encoding="utf-8") as f:
    f.writelines(lignes_md)

print(f"✅ Data dictionary exporté :")
print(f"   {chemin_json}")
print(f"   {chemin_md}")


✅ Data dictionary exporté :
   /home/jovyan/data/catalogue_donnees/data_dictionary.json
   /home/jovyan/data/catalogue_donnees/data_dictionary.md


---
## Étape 6 — Vérification : les métadonnées sont bien dans le catalogue Iceberg

In [7]:
print("=== DESCRIBE TABLE EXTENDED — bronze.livres ===")
spark.sql(f"DESCRIBE TABLE EXTENDED {NS_BRONZE}.livres").show(50, truncate=False)

print("\n=== Propriété 'comment' de la table ===")
proprietes = dict(spark.sql(f"SHOW TBLPROPERTIES {NS_BRONZE}.livres").collect())
print(proprietes.get("comment", "(aucun commentaire trouvé)"))


=== DESCRIBE TABLE EXTENDED — bronze.livres ===
+----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+
|col_name                    |data_type                                                                                                                                                                       |comment                                                       |
+----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+
|auteur                      |string                                                                                                       

In [8]:
spark.stop()
print("✅ SparkSession fermée — Catalogue enrichi via Groq !")


✅ SparkSession fermée — Catalogue enrichi via Groq !
